# Customer Support Ticket Synthetic Data Generator

## Objective
This project generates **synthetic customer support tickets** using Large Language Models (LLMs).

This generates structured customer support tickets containing:

- ticket_id
- customer_name
- product
- issue_category
- priority
- customer_message
- agent_response
- sentiment

The generator allows users to:
- create large volumes of support tickets
- compare outputs from different models
- export the generated dataset as CSV


In [71]:
import os
import json
import pandas as pd
import gradio as gr

from dotenv import load_dotenv
from openai import OpenAI
from transformers import pipeline


In [72]:
load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
HF_TOKEN = os.getenv("HF_TOKEN")

print("OpenRouter Key Loaded:", bool(OPENROUTER_API_KEY))
print("HF Token Loaded:", bool(HF_TOKEN))

OpenRouter Key Loaded: True
HF Token Loaded: True


In [73]:
openrouter = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

In [74]:
TICKET_SCHEMA = """
{
 "ticket_id": "string",
 "product": "string",
 "issue_category": "Billing | Technical | Account | Shipping | Refund",
 "priority": "Low | Medium | High | Critical",
 "customer_message": "string",
 "agent_response": "string"
}
"""

In [75]:
def build_prompt(num_tickets, product):

    return f"""
Generate {num_tickets} synthetic customer support tickets.

Return ONLY JSON objects, one per line.

Follow this schema:

{TICKET_SCHEMA}

Product: {product}
"""

In [76]:
def generate_openrouter(model, prompt, temperature=0.7):

    response = openrouter.chat.completions.create(
        model=model,
        messages=[
            {"role":"system","content":"You generate structured JSON support tickets"},
            {"role":"user","content":prompt}
        ],
        temperature=temperature
    )

    return response.choices[0].message.content

In [77]:
hf_generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device_map="auto",
    token=HF_TOKEN
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
def generate_huggingface(model_name, prompt, temperature=0.7):
    output = hf_generator(prompt, max_new_tokens=400, temperature=temperature, do_sample=True)
    return output[0]["generated_text"]


In [79]:
def parse_json_lines(text):

    records = []

    for line in text.split("\n"):
        line = line.strip()

        if line.startswith("{"):
            try:
                records.append(json.loads(line))
            except:
                pass

    return records


In [ ]:
#For HF the data outputted is raw not json
def generate_dataset(provider, model_name, product, num_tickets, temperature):
    prompt = build_prompt(num_tickets, product)
    
    if provider == "OpenRouter":
        raw_output = generate_openrouter(model_name, prompt, temperature)
        records = parse_json_lines(raw_output)
        return pd.DataFrame(records)
    else:
        raw_output = generate_huggingface(model_name, prompt, temperature)
        print(raw_output)
        return None

In [87]:
df_hf = generate_dataset(
    provider="HuggingFace",
    model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    product="Mobile Banking App",
    num_tickets=5,
    temperature=0.7
)
df_hf

Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generate 5 synthetic customer support tickets.

Return ONLY JSON objects, one per line.

Follow this schema:


{
 "ticket_id": "string",
 "product": "string",
 "issue_category": "Billing | Technical | Account | Shipping | Refund",
 "priority": "Low | Medium | High | Critical",
 "customer_message": "string",
 "agent_response": "string"
}


Product: Mobile Banking App
Issue Category: Technical
Priority: High
Customer Message: Error on login, unable to connect to server.
Agent Response: Please try again later or contact technical support.

Product: Online Banking App
Issue Category: Technical
Priority: Medium
Customer Message: Error while making a transfer. Please try again later.
Agent Response: Please check your internet connection and try again.

Product: Mobile Banking App
Issue Category: Technical
Priority: Critical
Customer Message: Error while accessing account. Please contact technical support.
Agent Response: Please try again later or contact technical support.

Product: Online 

In [83]:
df = generate_dataset(
    provider="OpenRouter",
    model_name="openai/gpt-oss-120b",
    product="Mobile Banking App",
    num_tickets=10,
    temperature=0.7
)

df

,ticket_id,product,issue_category,priority,customer_message,agent_response
0,TKT1001,Mobile Banking App,Billing,High,I was charged twice for my grocery purchase th...,We’re sorry for the inconvenience. Our billing...
1,TKT1002,Mobile Banking App,Technical,Critical,The app crashes every time I try to open the '...,Thank you for reporting this. Our engineers ha...
2,TKT1003,Mobile Banking App,Account,Medium,I can't log in after changing my password; it ...,We’ve unlocked your account. Please try loggin...
3,TKT1004,Mobile Banking App,Shipping,Low,When will the new debit card I ordered arrive?...,Your card was dispatched on March 1st via stan...
4,TKT1005,Mobile Banking App,Refund,High,"I cancelled a subscription last week, but I st...",We’ve processed a full refund for the subscrip...
5,TKT1006,Mobile Banking App,Technical,Medium,Push notifications for transaction alerts are ...,Please ensure notifications are enabled for th...
6,TKT1007,Mobile Banking App,Account,Low,How can I update my mailing address for statem...,You can update your mailing address under Sett...
7,TKT1008,Mobile Banking App,Billing,Critical,My overseas transaction was declined and I was...,We’ve waived the failed‑transaction fee and ar...
8,TKT1009,Mobile Banking App,Refund,Medium,"I returned a purchase made through the app, bu...",We’ve reopened the refund request with the mer...
9,TKT1010,Mobile Banking App,Shipping,Low,Can I expedite the delivery of my replacement ...,"Yes, we can upgrade to express shipping for an..."
